# Introduction

The common approach for deploying Scikit-Learn models on PySpark is wrapping them into Python UDFs. 

This approach has notable drawbacks:
* Opacity. The model is an executable black box. It takes effort to verify a setup before its predictions can be trusted.
* Dependency hell. All Spark executors must have exactly the same set of Python libraries installed and configured.
* Serialization overhead. Python UDFs shuttle data between JVM and Python processes.

The [MLflow](https://mlflow.org) framework aims to relieve the situation by standardizing model execution APIs and environments. However, it is unable to address deeper technological incompatibilities.

The [JPMML-MLflow](https://github.com/jpmml/jpmml-mlflow) project builds on top of MLflow, to minimize or break Python dependence in areas where it hurts the most:

* Transparency. The Predictive Model Markup Language (PMML) flavor is a text-based representation of model schema and logic. It is saved next to existing binary flavor(s), making the model fully transparent to humans and AI agents. 
* Plain PySpark. The PMML flavor is fully decoupled from the model training environment. No need to install or configure third-party Python libraries.
* Native JVM scoring. The PMML flavor can be loaded as a Java UDF that runs directly on Spark data inside the Spark executor.

# Installation

## Local

```bash
pip install jpmml-mlflow[evaluator-spark,sklearn,xgboost] seaborn
```

## Google Colab

In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Installing...")
    !{sys.executable} -m pip install --quiet jpmml-mlflow[evaluator-spark,sklearn,xgboost] seaborn
    print("Installation complete")

# Dataset

Use a sparse dataset, with both continuous and categorical features.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category = FutureWarning)

import numpy
import pandas
import seaborn

df = seaborn.load_dataset("titanic")

cat_cols = ["pclass", "sex", "embarked"]
cont_cols = ["age", "sibsp", "parch", "fare"]

X = df[cat_cols + cont_cols]
y = df["survived"]

print(X.head(10))

# Scikit-Learn training

The pipeline consists of a column mapper step, followed by a classifier step.

Cast categorical columns to `pandas.CategoricalDtype` data type.
Newer XGBoost versions support direct splitting on categorical values, so there is no need for one-hot encoding.
Cast continuous columns to `float32` data type.

Missing values flow through untouched as `pandas.NA` and `float("NaN")` values, respectively.
XGBoost supports missing values, so there is no need for imputation.

Switch the pipeline from the default NumPy mode to Pandas mode, so that `pandas.CategoricalDtype` columns do not get downcasted as the data flows between pipeline steps.

In [ ]:
from pandas import CategoricalDtype
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn2pmml.decoration import CategoricalDomain, ContinuousDomain
from xgboost import XGBClassifier

def make_titanic_pipeline():
    transformer = ColumnTransformer(
        [(cat_col, CategoricalDomain(dtype = CategoricalDtype()), [cat_col]) for cat_col in cat_cols] +
        [(cont_col, ContinuousDomain(dtype = numpy.float32), [cont_col]) for cont_col in cont_cols]
    )
    classifier = XGBClassifier(n_estimators = 31, enable_categorical = True, random_state = 42)
    
    pipeline = Pipeline([
        ("transformer", transformer),
        ("classifier", classifier)
    ])
    pipeline.set_output(transform = "pandas")

    return pipeline

Do a test conversion to see if the pipeline is fully PMML compatible.

The target field appears as "y" in the generated PMML document, because Scikit-Learn does not record label name(s).

In [ ]:
from sklearn2pmml import sklearn2pmml

pipeline = make_titanic_pipeline()
pipeline.fit(X, y)

yt_proba = pipeline.predict_proba(X)
print(yt_proba)

sklearn2pmml(pipeline, "titanic.pmml")

# MLflow logging

Upgrade from plain Scikit-Learn models to PMML-flavored ones by replacing the `mlflow.sklearn` module prefix with the `jpmml_mlflow.sklearn` module prefix.

Always pass as much model metadata as possible to the `log_model` function call.

The `signature` argument should hold the intended model input and result field names. The `input_example` argument should hold a small but representative dataset for model self-verification purposes.

In [ ]:
from mlflow.models import infer_signature

import jpmml_mlflow.sklearn
import mlflow
#import mlflow.sklearn

signature = infer_signature(X, y)
input_example = X.sample(n = 20, random_state = 42)

with mlflow.start_run() as run:
    pipeline = make_titanic_pipeline()
    pipeline.fit(X, y)
    
    jpmml_mlflow.sklearn.log_model(pipeline, name = "titanic", signature = signature, input_example = input_example)
    #mlflow.sklearn.log_model(pipeline, name = "titanic", signature = signature, input_example = input_example)

model_uri = "runs:/" + run.info.run_id + "/titanic"
print(model_uri)

Load the PMML flavor.

The target field should now appear with the correct name (ie. "survived" instead of "y").

In [ ]:
import jpmml_mlflow.pmml

pmml_bytes = jpmml_mlflow.pmml.load_model(model_uri)
pmml_str = pmml_bytes.decode("utf-8")

print("Size of model in PMML flavor: {} bytes".format(len(pmml_bytes)))
#print(pmml_str[:5000])

# PySpark deployment

Create a PySpark session, which includes all [JPMML-Evaluator-Spark](https://github.com/jpmml/jpmml-evaluator-spark) library JAR files on its classpath.

In [ ]:
from pyspark.sql import SparkSession

import jpmml_mlflow.evaluator_spark
import pyspark

spark_jars = jpmml_mlflow.evaluator_spark.classpath(version = pyspark.__version__)

spark = SparkSession.builder \
    .config("spark.jars", ",".join(spark_jars)) \
    .getOrCreate()

sparkDf = spark.createDataFrame(X)
sparkDf.show()

Load the PMML flavor into a JPMML-Evaluator-Spark evaluator object, and wrap it into the most appropriate [JPMML-Evaluator-PySpark](https://github.com/jpmml/jpmml-evaluator-pyspark) transformer object.

Transform data as with any PySpark transformer.

In [ ]:
from jpmml_evaluator_pyspark import FlatPMMLTransformer, NestedPMMLTransformer

import jpmml_mlflow.evaluator_spark

evaluator = jpmml_mlflow.evaluator_spark.load_model(model_uri)

pmmlTransformer = FlatPMMLTransformer(evaluator)
#pmmlTransformer = NestedPMMLTransformer(evaluator)

pmmlDf = pmmlTransformer.transform(sparkDf)
pmmlDf.show()

All done!

In [ ]:
spark.stop()